In [ ]:
import os, sys, time, json, random, shutil, math
from pathlib import Path
from collections import defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.cuda.amp import GradScaler, autocast
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms as T
import torchvision.models as models
import torchvision.models as tv_models

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}, CUDA: {torch.version.cuda}")

In [ ]:
CFG = {
    "backbone": "resnet101_ibn_a",
    "feat_dim": 2048,
    "img_size": (384, 384),
    "gem_p": 3.0,
    "num_classes": 576,
    "num_cameras": 20,
    "epochs": 120,
    "batch_size": 32,
    "num_instances": 4,
    "lr": 3.5e-4,
    "optimizer": "adamw",
    "warmup_epochs": 10,
    "eta_min": 1e-6,
    "weight_decay": 1e-4,
    "label_smoothing": 0.1,
    "triplet_margin": 0.3,
    "triplet_weight": 1.0,
    "circle_weight": 0.0,
    "id_weight": 1.0,
    "random_erasing_prob": 0.5,
    "color_jitter": True,
    "eval_every": 10,
    "fp16": True,
}
print(f"Config: {json.dumps(CFG, indent=2)}")

In [ ]:
# VeRi-776 dataset structure:
# /kaggle/input/veri-vehicle-re-identification-dataset/VeRi/
#   image_train/ - training imgs, filename: XXXX_cYYY_NNNNNNNN_N.jpg
#   image_query/ - query imgs
#   image_test/  - gallery imgs

def parse_veri776(root):
    """Parse VeRi-776 dataset.
    Returns train, query, gallery lists of (path, pid, camid).
    Train pids are relabeled 0..N-1.
    """
    root = Path(root)

    def parse_split(split_dir, relabel=False):
        data = []
        pid_set = set()
        img_dir = root / split_dir
        assert img_dir.exists(), f"Split not found: {img_dir}"

        for img_path in sorted(img_dir.glob("*.jpg")):
            parts = img_path.stem.split("_")
            pid = int(parts[0])
            camid = int(parts[1][1:]) - 1
            pid_set.add(pid)
            data.append((str(img_path), pid, camid))

        if relabel:
            pid2label = {pid: idx for idx, pid in enumerate(sorted(pid_set))}
            data = [(p, pid2label[pid], c) for p, pid, c in data]

        return data, len(pid_set)

    train, n_train_ids = parse_split("image_train", relabel=True)
    query, n_query_ids = parse_split("image_query", relabel=False)
    gallery, n_gallery_ids = parse_split("image_test", relabel=False)

    print(f"VeRi-776 parsed:")
    print(f"  Train:   {len(train):,} images, {n_train_ids} IDs")
    print(f"  Query:   {len(query):,} images, {n_query_ids} IDs")
    print(f"  Gallery: {len(gallery):,} images, {n_gallery_ids} IDs")

    return train, query, gallery, n_train_ids

VERI_CANDIDATES = [
    "/kaggle/input/veri-vehicle-re-identification-dataset/VeRi",
    "/kaggle/input/veri-vehicle-re-identification-dataset",
    "/kaggle/input/datasets/abhyudaya12/veri-vehicle-re-identification-dataset/VeRi",
    "/kaggle/input/datasets/abhyudaya12/veri-vehicle-re-identification-dataset",
]
VERI_ROOT = None
for cand in VERI_CANDIDATES:
    if Path(cand).exists() and (Path(cand) / "image_train").exists():
        VERI_ROOT = cand
        break
assert VERI_ROOT, f"VeRi-776 not found! Tried: {VERI_CANDIDATES}"
print(f"VeRi root: {VERI_ROOT}")

train_data, query_data, gallery_data, NUM_CLASSES = parse_veri776(VERI_ROOT)
CFG["num_classes"] = NUM_CLASSES
print(f"Updated num_classes = {NUM_CLASSES}")

In [ ]:
class ReIDDataset(Dataset):
    def __init__(self, data, transform=None):
        self.data = data
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        path, pid, camid = self.data[idx]
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, pid, camid


class PKSampler(torch.utils.data.Sampler):
    """Sample P identities each with K instances per batch."""

    def __init__(self, data, p, k):
        self.p = p
        self.k = k
        self.pid2idxs = defaultdict(list)
        for i, (_, pid, _) in enumerate(data):
            self.pid2idxs[pid].append(i)
        self.pids = list(self.pid2idxs.keys())

    def __iter__(self):
        random.shuffle(self.pids)
        batch = []
        for pid in self.pids:
            idxs = self.pid2idxs[pid]
            if len(idxs) < self.k:
                selected = random.choices(idxs, k=self.k)
            else:
                selected = random.sample(idxs, self.k)
            batch.extend(selected)
            if len(batch) >= self.p * self.k:
                yield from batch[: self.p * self.k]
                batch = batch[self.p * self.k :]
        if batch:
            yield from batch

    def __len__(self):
        return len(self.pids) * self.k


H, W = CFG["img_size"]
train_transform = T.Compose([
    T.Resize((H, W), interpolation=T.InterpolationMode.BICUBIC),
    T.RandomHorizontalFlip(p=0.5),
    T.Pad(10),
    T.RandomCrop((H, W)),
    T.ColorJitter(brightness=0.2, contrast=0.15, saturation=0.1, hue=0.05),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    T.RandomErasing(p=CFG["random_erasing_prob"], value="random"),
])

val_transform = T.Compose([
    T.Resize((H, W), interpolation=T.InterpolationMode.BICUBIC),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

P = CFG["batch_size"] // CFG["num_instances"]
sampler = PKSampler(train_data, p=P, k=CFG["num_instances"])
train_loader = DataLoader(
    ReIDDataset(train_data, train_transform),
    batch_size=CFG["batch_size"],
    sampler=sampler,
    num_workers=2,
    pin_memory=True,
    drop_last=True,
)
query_loader = DataLoader(
    ReIDDataset(query_data, val_transform),
    batch_size=128, shuffle=False, num_workers=2, pin_memory=True,
)
gallery_loader = DataLoader(
    ReIDDataset(gallery_data, val_transform),
    batch_size=128, shuffle=False, num_workers=2, pin_memory=True,
)
print(f"Train: {len(train_loader)} batches, Query: {len(query_loader)}, Gallery: {len(gallery_loader)}")

In [ ]:
import os
import socket
import traceback
import urllib.request

IBN_NET_RESNET101_A_URL = "https://github.com/XingangPan/IBN-Net/releases/download/v1.0/resnet101_ibn_a-59ea0ac6.pth"
IBN_NET_RESNET101_A_LOCAL_PATH = "/kaggle/working/resnet101_ibn_a.pth"
STEP6_MODEL_ERROR_PATH = "/kaggle/working/STEP6_MODEL_ERROR.txt"


class IBN_a(nn.Module):
    def __init__(self, planes):
        super().__init__()
        half = planes // 2
        self.IN = nn.InstanceNorm2d(half, affine=True)
        self.BN = nn.BatchNorm2d(planes - half)

    def forward(self, x):
        split = x.shape[1] // 2
        return torch.cat([self.IN(x[:, :split]), self.BN(x[:, split:])], dim=1)


class GeM(nn.Module):
    def __init__(self, p=3.0, eps=1e-6):
        super().__init__()
        self.p = nn.Parameter(torch.ones(1) * p)
        self.eps = eps

    def forward(self, x):
        return F.adaptive_avg_pool2d(x.clamp(min=self.eps).pow(self.p), 1).pow(1.0 / self.p)


class ResNet101IBN(nn.Module):
    def __init__(self, num_classes=CFG["num_classes"], feat_dim=2048, last_stride=1, gem_p=3.0, pretrained=True):
        super().__init__()
        base = tv_models.resnet101(weights=None)
        for layer in [base.layer1, base.layer2, base.layer3]:
            for block in layer:
                if hasattr(block, "bn1"):
                    block.bn1 = IBN_a(block.bn1.num_features)
        load_result = None
        if pretrained:
            state_dict = None
            if os.path.exists(IBN_NET_RESNET101_A_LOCAL_PATH):
                print(f"Using cached pretrained weights: {IBN_NET_RESNET101_A_LOCAL_PATH}")
            else:
                try:
                    print(f"Downloading pretrained weights from {IBN_NET_RESNET101_A_URL}")
                    print(f"Saving pretrained weights to {IBN_NET_RESNET101_A_LOCAL_PATH}")
                    original_timeout = socket.getdefaulttimeout()
                    socket.setdefaulttimeout(120)
                    try:
                        urllib.request.urlretrieve(
                            IBN_NET_RESNET101_A_URL,
                            IBN_NET_RESNET101_A_LOCAL_PATH,
                        )
                    finally:
                        socket.setdefaulttimeout(original_timeout)
                    print(f"Pretrained weights downloaded to {IBN_NET_RESNET101_A_LOCAL_PATH}")
                except Exception as download_error:
                    print(f"Direct download failed: {download_error}")
                    print("Falling back to torch.hub.load_state_dict_from_url")
                    state_dict = torch.hub.load_state_dict_from_url(
                        IBN_NET_RESNET101_A_URL,
                        map_location="cpu",
                        progress=True,
                    )
            if state_dict is None:
                print(f"Loading pretrained weights from {IBN_NET_RESNET101_A_LOCAL_PATH}")
                state_dict = torch.load(IBN_NET_RESNET101_A_LOCAL_PATH, map_location="cpu")
            load_result = base.load_state_dict(state_dict, strict=False)
            print(f"Missing keys: {load_result.missing_keys}")
            print(f"Unexpected keys: {load_result.unexpected_keys}")
        if load_result is not None:
            assert set(load_result.missing_keys) <= {"fc.weight", "fc.bias"}, f"Unexpected missing keys: {load_result.missing_keys}"
            allowed_unexpected_keys = {"fc.weight", "fc.bias", "classifier.weight", "classifier.bias"}
            assert set(load_result.unexpected_keys) <= allowed_unexpected_keys, f"Unexpected extra keys: {load_result.unexpected_keys}"
        if last_stride == 1:
            for module in base.layer4.modules():
                if isinstance(module, nn.Conv2d) and module.stride == (2, 2):
                    module.stride = (1, 1)
        self.conv1 = base.conv1
        self.bn1 = base.bn1
        self.relu = base.relu
        self.maxpool = base.maxpool
        self.layer1 = base.layer1
        self.layer2 = base.layer2
        self.layer3 = base.layer3
        self.layer4 = base.layer4
        self.pool = GeM(p=gem_p)
        self.feat_dim = feat_dim
        self.bottleneck = nn.BatchNorm1d(feat_dim)
        self.bottleneck.bias.requires_grad_(False)
        nn.init.constant_(self.bottleneck.weight, 1)
        nn.init.constant_(self.bottleneck.bias, 0)
        self.classifier = nn.Linear(feat_dim, num_classes, bias=False)
        nn.init.normal_(self.classifier.weight, std=0.001)

    def forward(self, x):
        x = self.maxpool(self.relu(self.bn1(self.conv1(x))))
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        global_feat = self.pool(x).view(x.size(0), -1)
        bn_feat = self.bottleneck(global_feat)
        if self.training:
            return self.classifier(bn_feat), global_feat, bn_feat
        return F.normalize(bn_feat, p=2, dim=1)


def build_resnet101_ibn_a(num_classes, feat_dim, gem_p):
    return ResNet101IBN(
        num_classes=num_classes,
        feat_dim=feat_dim,
        last_stride=1,
        gem_p=gem_p,
        pretrained=True,
    )

In [ ]:
class CrossEntropyLabelSmooth(nn.Module):
    def __init__(self, num_classes, epsilon=0.1):
        super().__init__()
        self.num_classes = num_classes
        self.epsilon = epsilon
        self.logsoftmax = nn.LogSoftmax(dim=1)

    def forward(self, inputs, targets):
        log_probs = self.logsoftmax(inputs)
        targets_one_hot = torch.zeros_like(log_probs).scatter_(1, targets.unsqueeze(1), 1)
        targets_one_hot = (1 - self.epsilon) * targets_one_hot + self.epsilon / self.num_classes
        loss = (-targets_one_hot * log_probs).mean(0).sum()
        return loss


class TripletLoss(nn.Module):
    def __init__(self, margin=0.3):
        super().__init__()
        self.margin = margin
        self.ranking_loss = nn.MarginRankingLoss(margin=margin)

    def forward(self, feats, labels):
        dist = torch.cdist(feats, feats, p=2)
        mask_pos = labels.unsqueeze(0) == labels.unsqueeze(1)
        dist_ap = (dist * mask_pos.float()).max(dim=1)[0]
        dist_an = (dist + mask_pos.float() * 1e9).min(dim=1)[0]
        y = torch.ones_like(dist_ap)
        return self.ranking_loss(dist_an, dist_ap, y)


id_criterion = CrossEntropyLabelSmooth(CFG["num_classes"], epsilon=CFG["label_smoothing"]).to(DEVICE)
triplet_criterion = TripletLoss(margin=CFG["triplet_margin"]).to(DEVICE)
print("Losses: ID (label smooth) + Triplet (hard mining)")

In [ ]:
model = build_resnet101_ibn_a(CFG["num_classes"], CFG["feat_dim"], CFG["gem_p"]).to(DEVICE)
print(f"Model params: {sum(p.numel() for p in model.parameters()):,}")

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=CFG["lr"],
    weight_decay=CFG["weight_decay"],
)

warmup_epochs = CFG["warmup_epochs"]
total_epochs = CFG["epochs"]

def lr_lambda(epoch):
    if epoch < warmup_epochs:
        return 0.01 + (1.0 - 0.01) * epoch / warmup_epochs
    progress = (epoch - warmup_epochs) / (total_epochs - warmup_epochs)
    return max(CFG["eta_min"] / CFG["lr"], 0.5 * (1 + math.cos(math.pi * progress)))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
scaler = GradScaler(enabled=CFG["fp16"])
print(f"Optimizer: AdamW, LR={CFG['lr']}, WD={CFG['weight_decay']}")

In [ ]:
@torch.no_grad()
def extract_features(model, loader):
    model.eval()
    feats, pids, camids = [], [], []
    for imgs, pid, camid in loader:
        imgs = imgs.to(DEVICE)
        feat = model(imgs)
        feats.append(feat.cpu())
        pids.extend(pid.tolist())
        camids.extend(camid.tolist())
    return torch.cat(feats, 0), np.array(pids), np.array(camids)


def eval_reid(qf, q_pids, q_camids, gf, g_pids, g_camids):
    """Compute mAP and CMC (Rank-1, Rank-5, Rank-10)."""
    m, n = qf.shape[0], gf.shape[0]
    distmat = torch.cdist(qf, gf, p=2).numpy()
    indices = np.argsort(distmat, axis=1)

    all_ap = []
    all_cmc = np.zeros(n)
    num_valid = 0

    for i in range(m):
        q_pid = q_pids[i]
        q_camid = q_camids[i]
        order = indices[i]
        g_pid_sorted = g_pids[order]
        g_camid_sorted = g_camids[order]

        valid = ~((g_pid_sorted == q_pid) & (g_camid_sorted == q_camid))
        valid &= g_pid_sorted != -1

        if not np.any(g_pid_sorted[valid] == q_pid):
            continue

        g_pid_valid = g_pid_sorted[valid]
        matches = (g_pid_valid == q_pid).astype(np.float32)

        first_match = np.where(matches == 1)[0]
        if len(first_match) > 0:
            all_cmc[first_match[0]:] += 1

        num_rel = matches.sum()
        cum_matches = np.cumsum(matches)
        precision = cum_matches / (np.arange(len(matches)) + 1)
        ap = (precision * matches).sum() / num_rel
        all_ap.append(ap)
        num_valid += 1

    cmc = all_cmc / num_valid
    mAP = np.mean(all_ap)
    return mAP, cmc[0], cmc[4], cmc[9]


def evaluate(model, query_loader, gallery_loader):
    qf, q_pids, q_camids = extract_features(model, query_loader)
    gf, g_pids, g_camids = extract_features(model, gallery_loader)
    mAP, r1, r5, r10 = eval_reid(qf, q_pids, q_camids, gf, g_pids, g_camids)
    return mAP, r1, r5, r10

In [ ]:
OUTPUT_DIR = Path("/kaggle/working/checkpoints")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

best_mAP = 0.0
history = []

for epoch in range(total_epochs):
    model.train()
    running_loss = 0.0
    running_id = 0.0
    running_tri = 0.0
    n_batches = 0
    t0 = time.time()

    for imgs, pids_batch, _ in train_loader:
        imgs = imgs.to(DEVICE)
        pids_batch = pids_batch.to(DEVICE)

        optimizer.zero_grad()
        with autocast(enabled=CFG["fp16"]):
            cls_score, global_feat, bn_feat = model(imgs)
            loss_id = id_criterion(cls_score, pids_batch) * CFG["id_weight"]
            loss_tri = triplet_criterion(global_feat, pids_batch) * CFG["triplet_weight"]
            loss = loss_id + loss_tri

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()
        running_id += loss_id.item()
        running_tri += loss_tri.item()
        n_batches += 1

    scheduler.step()
    elapsed = time.time() - t0
    avg_loss = running_loss / max(n_batches, 1)
    avg_id = running_id / max(n_batches, 1)
    avg_tri = running_tri / max(n_batches, 1)
    lr_now = scheduler.get_last_lr()[0]

    print(f"E{epoch:03d} | Loss={avg_loss:.4f} (ID={avg_id:.4f} Tri={avg_tri:.4f}) | LR={lr_now:.6f} | {elapsed:.0f}s")

    if (epoch + 1) % CFG["eval_every"] == 0 or epoch == total_epochs - 1:
        mAP, r1, r5, r10 = evaluate(model, query_loader, gallery_loader)
        print(f"  >> mAP={mAP:.4f}  R1={r1:.4f}  R5={r5:.4f}  R10={r10:.4f}")

        history.append({
            "epoch": epoch, "mAP": float(mAP), "R1": float(r1),
            "R5": float(r5), "R10": float(r10),
            "loss": avg_loss, "lr": lr_now,
        })

        ckpt_path = OUTPUT_DIR / f"ckpt_e{epoch:02d}.pth"
        torch.save({
            "epoch": epoch,
            "mAP": float(mAP),
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
        }, ckpt_path)

        if mAP > best_mAP:
            best_mAP = mAP
            best_path = OUTPUT_DIR / "resnet101ibn_veri776_best.pth"
            torch.save(model.state_dict(), best_path)
            print(f"  * New best! mAP={best_mAP:.4f} saved to {best_path.name}")

print(f"\nTraining complete. Best mAP={best_mAP:.4f}")

In [ ]:
best_path = OUTPUT_DIR / "resnet101ibn_veri776_best.pth"
model.load_state_dict(torch.load(best_path, map_location=DEVICE))
mAP, r1, r5, r10 = evaluate(model, query_loader, gallery_loader)
print(f"Final eval: mAP={mAP:.4f}  R1={r1:.4f}  R5={r5:.4f}  R10={r10:.4f}")

export_path = Path("/kaggle/working/resnet101ibn_veri776_best.pth")
shutil.copy2(best_path, export_path)
print(f"Exported: {export_path} ({export_path.stat().st_size / 1024**2:.1f} MB)")

hist_path = Path("/kaggle/working/training_history_veri776.json")
with open(hist_path, "w") as f:
    json.dump({"config": CFG, "history": history, "best_mAP": best_mAP}, f, indent=2)
print(f"History: {hist_path}")

meta = {
    "model": "resnet101_ibn_a",
    "dataset": "VeRi-776",
    "best_mAP": float(best_mAP),
    "best_R1": float(r1),
    "feat_dim": CFG["feat_dim"],
    "img_size": CFG["img_size"],
    "num_classes": CFG["num_classes"],
    "epochs": CFG["epochs"],
    "purpose": "VeRi-776 pretraining for CityFlowV2 fine-tuning (09d)",
}
meta_path = Path("/kaggle/working/veri776_metadata.json")
with open(meta_path, "w") as f:
    json.dump(meta, f, indent=2)
print(f"Metadata: {meta_path}")